# Waste Type Identification — Sanity check *multi-oggetto della stessa classe* (SOLO VALUTAZIONE)

## 1. Configurazione

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time
from pathlib import Path

# Percorsi
BASE              = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
RESULTS_DIR       = BASE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Iperparametri
IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_CLASSES = 8
SEED        = 1234

CLASS_NAMES = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']

# Modello da verificare
ARCH         = 'resnet18'
WEIGHTS_PATH = MODELS_DIR / 'resnet18_acq_mild.pth'

# Parametri del check
GRIDS                = [2, 3]                                                   # mosaici 2x2 e 3x3
N_MONTAGES_PER_CLASS = 40
CELL, GUTTER, MARGIN = 200, 14, 18

print('Modello da verificare:', ARCH, '|', WEIGHTS_PATH.name)

Mounted at /content/drive
Modello da verificare: resnet18 | resnet18_acq_mild.pth


## 2. Copia locale del dataset

In [ ]:
import shutil
if not DATASET_DIR.exists():
    print('Copio il dataset in locale, attendi qualche minuto...')
    t0 = time.time()
    shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Copia completata in {time.time() - t0:.0f}s')
else:
    print('Copia locale gia presente:', DATASET_DIR)

Copio il dataset in locale, attendi qualche minuto...
Copia completata in 686s


## 3. Validation set e preprocessing

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
import random

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

df_all = pd.read_csv(SPLIT_CSV)
df_val = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f'Immagini di validazione: {len(df_val)}')
print(df_val['macro_label'].value_counts().sort_index())

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

Device: cuda
Immagini di validazione: 3102
macro_label
battery              189
clothing            1460
glass                402
metal                154
organic              197
papery               388
plastic              173
undifferentiated     139
Name: count, dtype: int64


## 4. Caricamento del modello (solo inferenza)

In [ ]:
def build_model(arch):
    if arch == 'resnet18':
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif arch == 'regnet_y_1_6gf':
        m = models.regnet_y_1_6gf(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    else:
        raise ValueError(f'arch non supportata: {arch}')
    return m

def load_model(arch, weights_path):
    m = build_model(arch)
    state = torch.load(weights_path, map_location='cpu')
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state)
    return m.to(device).eval()

model = load_model(ARCH, WEIGHTS_PATH)
print('Pesi caricati da:', WEIGHTS_PATH)

Pesi caricati da: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/models/resnet18_acq_mild.pth


## 5. Costruzione dei mosaici della stessa classe

In [ ]:
def make_montage(paths, grid):
    side = MARGIN * 2 + grid * CELL + (grid - 1) * GUTTER
    canvas = Image.new('RGB', (side, side), (255, 255, 255))
    for k, p in enumerate(paths):
        r, c = divmod(k, grid)
        tile = Image.open(p).convert('RGB').resize((CELL, CELL), Image.BILINEAR)
        x = MARGIN + c * (CELL + GUTTER)
        y = MARGIN + r * (CELL + GUTTER)
        canvas.paste(tile, (x, y))
    return canvas

def build_class_montages(df, grid, n_per_class, seed):
    rng = np.random.RandomState(seed)
    items = []
    by_label = {lab: df[df['label'] == lab]['filepath'].tolist() for lab in range(NUM_CLASSES)}
    for lab, files in by_label.items():
        if len(files) == 0:
            continue
        k = grid * grid
        for _ in range(n_per_class):
            replace = len(files) < k
            idx = rng.choice(len(files), size=k, replace=replace)
            paths = [DATASET_DIR / files[j] for j in idx]
            items.append((make_montage(paths, grid), lab))
    return items

# Anteprima: salva un esempio per un paio di classi
for cls in ['plastic', 'metal']:
    lab = CLASS_NAMES.index(cls)
    files = df_val[df_val['label'] == lab]['filepath'].tolist()
    if len(files) >= 9:
        ex = make_montage([DATASET_DIR / files[j] for j in range(9)], 3)
        out = RESULTS_DIR / f'example_montage_3x3_{cls}.png'
        ex.save(out); print('Salvato esempio:', out)

Salvato esempio: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/results/example_montage_3x3_plastic.png
Salvato esempio: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/results/example_montage_3x3_metal.png


## 6. Valutazione

In [ ]:
class ImgItemsDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        obj, lab = self.items[i]
        img = obj if isinstance(obj, Image.Image) else Image.open(DATASET_DIR / obj).convert('RGB')
        return preprocess(img), lab

@torch.no_grad()
def predict_items(items, desc):
    dl = DataLoader(ImgItemsDataset(items), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in tqdm(dl, desc=desc, leave=False):
        ps.append(model(x.to(device)).argmax(1).cpu().numpy())
        ys.append(np.asarray(y))
    return np.concatenate(ys), np.concatenate(ps)

def per_class_and_balacc(y, p):
    accs = {}
    for lab in range(NUM_CLASSES):
        mask = (y == lab)
        accs[CLASS_NAMES[lab]] = float((p[mask] == lab).mean()) if mask.sum() > 0 else np.nan
    vals = [v for v in accs.values() if v == v]
    return accs, float(np.mean(vals))

single_items = list(zip(df_val['filepath'].tolist(), df_val['label'].tolist()))
y, p = predict_items(single_items, 'single')
single_acc, single_bal = per_class_and_balacc(y, p)
print(f'Singolo oggetto  -> balanced acc = {single_bal:.4f}')

montage_results = {}
for g in GRIDS:
    items = build_class_montages(df_val, g, N_MONTAGES_PER_CLASS, SEED + g)
    y, p = predict_items(items, f'{g}x{g}')
    accs, bal = per_class_and_balacc(y, p)
    montage_results[g] = (accs, bal)
    print(f'Mosaico {g}x{g}    -> balanced acc = {bal:.4f}  (su {len(items)} mosaici)')

single:   0%|          | 0/49 [00:00<?, ?it/s]

Singolo oggetto  -> balanced acc = 0.9652


2x2:   0%|          | 0/5 [00:00<?, ?it/s]

Mosaico 2x2    -> balanced acc = 0.9250  (su 320 mosaici)


3x3:   0%|          | 0/5 [00:00<?, ?it/s]

Mosaico 3x3    -> balanced acc = 0.6938  (su 320 mosaici)


## 7. Tabella riassuntiva e salvataggio

In [ ]:
table = {'single': single_acc}
for g in GRIDS:
    table[f'{g}x{g}'] = montage_results[g][0]

df_out = pd.DataFrame(table).round(4)
df_out.loc['BALANCED'] = [single_bal] + [montage_results[g][1] for g in GRIDS]
print(df_out)

print('\nDelta balanced accuracy rispetto al singolo oggetto (>= 0 = nessun peggioramento):')
for g in GRIDS:
    print(f'  {g}x{g}: {montage_results[g][1] - single_bal:+.4f}')

out_csv = RESULTS_DIR / f'multiobject_same_class_{ARCH}.csv'
df_out.to_csv(out_csv)
print('\nSalvato:', out_csv)

                    single    2x2      3x3
battery           0.989400  1.000  0.97500
clothing          0.997300  0.775  0.15000
glass             0.962700  1.000  0.97500
metal             0.935100  0.875  0.37500
organic           0.979700  0.925  0.67500
papery            0.984500  1.000  1.00000
plastic           0.901700  0.825  0.47500
undifferentiated  0.971200  1.000  0.92500
BALANCED          0.965202  0.925  0.69375

Delta balanced accuracy rispetto al singolo oggetto (>= 0 = nessun peggioramento):
  2x2: -0.0402
  3x3: -0.2715

Salvato: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/results/multiobject_same_class_resnet18.csv
